In [1]:
import pandas as pd

# Load the TSV file
file_path = "/home/atoffano/PFP_baselines/data/swissprot/2024_01/diamond_swissprot_2024_01_alignment.tsv"
df = pd.read_csv(file_path, sep='\t', header=None)

# Isolate ten unique entries in column 0
unique_entries = df[0].unique()[:10]

# Get all rows relating to those entries
filtered_df = df[df[0].isin(unique_entries)]

# Output the names to a txt file
with open("unique_names.txt", "w") as f:
    for name in unique_entries:
        f.write(f"{name}\n")

In [ ]:
import pandas as pd
import h5py
import numpy as np

# --- Step 1: Load and merge annotation files, then sample targets ---
ann_files = [
    "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_2024_01_MFO_exp_annotations.tsv",
    "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_2024_01_CCO_exp_annotations.tsv",
    "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_2024_01_BPO_exp_annotations.tsv"
]
dfs = [pd.read_csv(f, sep='\t', dtype=str) for f in ann_files]
merged = pd.concat(dfs, ignore_index=True)
all_proteins = merged['EntryID'].unique()
np.random.seed(42)  # For reproducibility
sample_size = 10 # Change as needed
targets = set(np.random.choice(all_proteins, size=sample_size, replace=False))

# --- Step 2: Filter diamond alignment for rows with targets in col 0 or 1 ---
align_path = "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/diamond_swissprot_2024_01_alignment.tsv"
df_align = pd.read_csv(align_path, sep='\t', header=None, dtype=str)
filtered = df_align[df_align[0].isin(targets) | df_align[1].isin(targets)]
unique_proteins = set(filtered[0]).union(filtered[1])
# Save aligment file
filtered.to_csv("/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/toy/toy_swissprot_2024_01_alignment.tsv", sep='\t', index=False, header=False)

# --- Step 3: Extract embeddings for those proteins from HDF5 ---
h5_in = "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_esm1b_per_aa.h5"
h5_out = "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/toy/toy_swissprot_esm1b_per_aa.h5"

with h5py.File(h5_in, "r") as fin, h5py.File(h5_out, "w") as fout:
    for pid in unique_proteins:
        if pid in fin:
            grp = fout.create_group(pid)
            fin[pid].copy("embeddings", grp)

# --- Step 4: Filter annotation files (CCO, MFO, BPO) and split into train/val/test ---
for aspect in ["CCO", "MFO", "BPO"]:
    ann_in = f"/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_2024_01_{aspect}_exp_annotations.tsv"
    df_ann = pd.read_csv(ann_in, sep='\t', dtype=str)
    filtered_ann = df_ann[df_ann['EntryID'].isin(unique_proteins)]
    aspect_proteins = filtered_ann['EntryID'].unique()
    np.random.seed(42)  # For reproducibility
    shuffled = np.random.permutation(aspect_proteins)
    test_p = set(shuffled[:2])
    val_p = set(shuffled[2:4])
    train_p = set(shuffled[4:])
    train_ann = filtered_ann[filtered_ann['EntryID'].isin(train_p)]
    val_ann = filtered_ann[filtered_ann['EntryID'].isin(val_p)]
    test_ann = filtered_ann[filtered_ann['EntryID'].isin(test_p)]
    train_out = f"/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/toy/toy_swissprot_2024_01_{aspect}_train_exp_annotations.tsv"
    val_out = f"/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/toy/toy_swissprot_2024_01_{aspect}_val_exp_annotations.tsv"
    test_out = f"/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/toy/toy_swissprot_2024_01_{aspect}_test_exp_annotations.tsv"
    train_ann.to_csv(train_out, sep='\t', index=False)
    val_ann.to_csv(val_out, sep='\t', index=False)
    test_ann.to_csv(test_out, sep='\t', index=False)


# --- Step 5: Filter InterPro file ---
ipr_in = "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_interpro_106_0.tsv"
ipr_out = "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/toy/toy_swissprot_interpro_106_0.tsv"
df_ipr = pd.read_csv(ipr_in, sep='\t', dtype=str)
filtered_ipr = df_ipr[df_ipr['ID'].isin(unique_proteins)]
filtered_ipr.to_csv(ipr_out, sep='\t', index=False)

In [ ]:
with h5py.File(h5_out, "r") as f:
    print("Groups in h5_out:", list(f.keys()))
    for pid in f.keys():
        print(f"Protein ID: {pid}")
        print("  Datasets:", list(f[pid].keys()))
        print("  Embeddings shape:", f[pid]["embeddings"].shape)